# Space Invader mosaic — DINOv2 contrastive (InfoNCE) — exp23

*Approach change from exp22's triplet margin loss.*

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `colab_reference_data.zip` to Drive root *(first time / on hash change)*
3. Upload `colab_flash_data.zip` to Drive root *(every run)*
4. Run all cells in order

**What changed vs exp22 (and why):**
- **Loss → InfoNCE / NT-Xent with in-batch negatives.** Each anchor is contrasted against every other item in the batch at once (~B−1 negatives for free), which is far more stable at a plateau than triplet margin and removes the dependence on hand-picked negatives. Same-invader collisions in a batch are masked so they aren't treated as false negatives.
- **Projection head by default** (`PROJ_HEAD=True`): the DINOv2 backbone is frozen and a small MLP (384→512→256) learns the metric. This avoids the capacity/plasticity problem of nudging 2 transformer blocks at LR 1e-5. Set `PROJ_HEAD=False` to fine-tune the last `UNFREEZE_N` blocks instead.
- **Hard negatives are phased in** after `HARDNEG_START_EPOCH` and **cleaned**: a verify-UI rejection is dropped if its reference embedding is within `HARDNEG_DROP_COS` cosine of the positive (likely a near-twin or mislabel — the thing that made exp22 regress).
- **Checkpoint on top-1 retrieval recall**, not triplet val loss. Recall on held-out flash images is what you actually care about; triplet loss can fall while recall stagnates.

Tune first: `TEMPERATURE` (0.05–0.1), `BATCH_SIZE` (as large as the T4 allows — more negatives is better), `PROJ_DIM`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
from pathlib import Path

REF_ZIP   = '/content/drive/MyDrive/colab_reference_data.zip'
FLASH_ZIP = '/content/drive/MyDrive/colab_flash_data.zip'
DATA_DIR  = '/content/data'

ref_sentinel = DATA_DIR + '/.ref_extracted'
if not os.path.exists(ref_sentinel):
    print('Extracting reference data (~1.4 GB, once per runtime)...')
    with zipfile.ZipFile(REF_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    Path(ref_sentinel).touch()
    print('Reference data ready')
else:
    print('Reference data already extracted, skipping')

print('Extracting flash data...')
with zipfile.ZipFile(FLASH_ZIP, 'r') as zf:
    zf.extractall(DATA_DIR)
print('Flash data ready')

In [ ]:
!pip install -q transformers albumentations ultralytics

In [ ]:
# ─────────────────────────────────────────────────────────────────
# exp23 — InfoNCE / contrastive with in-batch negatives + retrieval eval
#
# Key changes vs exp22:
#  * Loss: InfoNCE (NT-Xent) with in-batch negatives instead of triplet margin.
#    Every other item in the batch is a negative, so each anchor sees ~B-1
#    negatives for free. Same-invader collisions are masked out (false-neg fix).
#  * Optional projection head (PROJ_HEAD=True): freeze the backbone entirely
#    and learn a small MLP metric. More stable than nudging 2 blocks at 1e-5.
#  * Hard negatives from the verify UI are phased in (HARDNEG_START_EPOCH) and
#    cleaned (dropped if too close to the positive => likely near-twin/mislabel).
#  * Checkpoint on top-1 retrieval recall, not triplet val loss.
# ─────────────────────────────────────────────────────────────────

import json, random, time
from pathlib import Path

import albumentations as A
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel

# ── Configuration ────────────────────────────────────────────────
IMAGE_ROOT      = Path('/content/data/images')
FLASH_ROOT      = Path('/content/data/flash_images')
MANIFEST        = Path('/content/data/reference_manifest.jsonl')
FLASH_LABELS    = Path('/content/data/flash_labels.jsonl')
DETECTOR_PATH   = Path('/content/data/mosaic_detector_v4.pt')
EXP_NAME        = 'exp23'
OUTPUT_DIR      = Path(f'/content/drive/MyDrive/si-experiments/{EXP_NAME}')
WARMSTART_FROM  = ''            # '' = base model

BASE_MODEL      = 'facebook/dinov2-small'
TOTAL_EPOCHS    = 40
BATCH_SIZE      = 64            # bigger batch = more in-batch negatives = better InfoNCE
LR              = 3e-4          # head LR (higher; head trains from scratch)
BACKBONE_LR     = 1e-5          # only used if PROJ_HEAD=False
TEMPERATURE     = 0.07          # InfoNCE temperature
PROJ_HEAD       = True          # True: freeze backbone, train MLP head (recommended)
PROJ_DIM        = 256
UNFREEZE_N      = 4             # only used if PROJ_HEAD=False
FLASH_OVERSAMPLE = 8
HARDNEG_START_EPOCH = 8         # phase verify-UI hard negs in only after this epoch
HARDNEG_MAX_PER_IMG = 3         # cap to stop a few noisy negs dominating
HARDNEG_DROP_COS    = 0.90      # drop a "hard neg" if its ref is this close to the positive
VAL_FRACTION    = 0.15

NOTEBOOK_VERSION = '2026-06-14 — exp23, InfoNCE in-batch negs, projection head, recall eval'
# ─────────────────────────────────────────────────────────────────

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('=' * 60)
print(f'Notebook version: {NOTEBOOK_VERSION}')
print(f'Device:           {device}')
print(f'Mode:             {"projection head (frozen backbone)" if PROJ_HEAD else f"finetune last {UNFREEZE_N} blocks"}')
print('=' * 60)


FLASH_AUG = A.Compose([
    A.ColorJitter(brightness=0.6, contrast=0.6, saturation=0.4, hue=0.1, p=0.9),
    A.GaussianBlur(blur_limit=(3, 7), sigma_limit=(0.5, 3.0), p=0.6),
    A.Perspective(scale=(0.05, 0.15), p=0.5),
    A.Affine(rotate=(-20, 20), translate_percent=(-0.1, 0.1), scale=(0.75, 1.25), p=0.6),
    A.RandomBrightnessContrast(p=0.4),
    A.Sharpen(alpha=(0, 0.5), lightness=(0.5, 1.0), p=0.3),
    A.ToGray(p=0.05),
    A.ImageCompression(quality_range=(60, 95), p=0.3),
])
LIGHT_AUG = A.Compose([
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])


def open_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert('RGB')


class MosaicDetector:
    def __init__(self, model_path, conf=0.25):
        from ultralytics import YOLO
        self._model = YOLO(str(model_path))
        self._conf  = conf

    def crop(self, image_path):
        img = open_image(image_path)
        results = self._model(img, conf=self._conf, verbose=False)
        boxes = results[0].boxes
        if boxes is None or len(boxes) == 0:
            return None
        best = int(boxes.conf.argmax())
        x1, y1, x2, y2 = boxes.xyxy[best].tolist()
        return img.crop((x1, y1, x2, y2))


# ── Projection head ──────────────────────────────────────────────
class ProjectionHead(nn.Module):
    def __init__(self, in_dim=384, hidden=512, out_dim=PROJ_DIM):
        # dinov2-small CLS dim is 384
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def encode(backbone, head, pixel_values):
    """CLS token -> (optional head) -> L2-normalized embedding."""
    feats = backbone(pixel_values=pixel_values).last_hidden_state[:, 0]  # (B, 384)
    if head is not None:
        feats = head(feats)
    return F.normalize(feats, dim=-1)


# ── Contrastive dataset: returns (image_tensor, invader_id) ──────
class PairDataset(Dataset):
    """Yields (anchor_image, positive_image, invader_id).

    anchor   = a view of an invader (flash crop if flash item, else aug ref crop)
    positive = a *different* clean reference crop of the SAME invader
    invader_id is returned so the loss can mask same-invader in-batch collisions.

    No negatives are sampled here — InfoNCE draws them from the batch.
    """
    def __init__(self, by_id, processor, samples_per_invader=4, augment=False,
                 detector=None, flash_pairs=None, flash_oversample=8):
        self._by_id   = {k: v for k, v in by_id.items() if len(v) >= 2}
        self._ids     = list(self._by_id.keys())
        self._processor = processor
        self._augment = augment
        self._spi     = samples_per_invader
        self._ref_len = len(self._ids) * samples_per_invader
        self._flash_pairs = flash_pairs or []
        self._flash_oversample = flash_oversample
        self._flash_len = len(self._flash_pairs) * flash_oversample

        self._ref_crops = {}
        if detector is not None:
            all_paths = {p for paths in self._by_id.values() for p in paths}
            print(f'Pre-computing reference crops for {len(all_paths)} images...')
            for i, path in enumerate(sorted(all_paths), 1):
                if i % 200 == 0 or i == len(all_paths):
                    print(f'  {i}/{len(all_paths)}', end='\r')
                self._ref_crops[path] = detector.crop(IMAGE_ROOT / path)
            found = sum(1 for v in self._ref_crops.values() if v is not None)
            print(f'  Reference crops: {found}/{len(all_paths)} ({100*found//max(1,len(all_paths))}%)')

        self._flash_crops = {}
        if detector is not None and self._flash_pairs:
            print(f'Pre-computing flash crops for {len(self._flash_pairs)} images...')
            for path, _ in self._flash_pairs:
                self._flash_crops[path] = detector.crop(path)
            found = sum(1 for v in self._flash_crops.values() if v is not None)
            print(f'  Flash crops: {found}/{len(self._flash_pairs)} ({100*found//max(1,len(self._flash_pairs))}%)')

    def __len__(self):
        return self._ref_len + self._flash_len

    def _proc(self, img):
        return self._processor(images=img, return_tensors='pt')['pixel_values'].squeeze(0)

    def _load_ref(self, path, aug=None):
        img = self._ref_crops.get(path) or open_image(IMAGE_ROOT / path)
        if aug is not None:
            img = aug(image=np.array(img))['image']
        return self._proc(img)

    def _load_flash(self, path, aug=None):
        img = self._flash_crops.get(path) or open_image(path)
        if aug is not None:
            img = aug(image=np.array(img))['image']
        return self._proc(img)

    def __getitem__(self, idx):
        if idx < self._ref_len:
            inv_id = self._ids[idx % len(self._ids)]
            anchor_path, pos_path = random.sample(self._by_id[inv_id], 2)
            if self._augment:
                return self._load_ref(anchor_path, FLASH_AUG), self._load_ref(pos_path, LIGHT_AUG), inv_id
            return self._load_ref(anchor_path), self._load_ref(pos_path), inv_id
        else:
            flash_path, inv_id = self._flash_pairs[(idx - self._ref_len) % len(self._flash_pairs)]
            pos_options = self._by_id.get(inv_id)
            if not pos_options:
                return self.__getitem__(idx % max(1, self._ref_len))
            pos_path = random.choice(pos_options)
            return self._load_flash(flash_path, LIGHT_AUG), self._load_ref(pos_path), inv_id


def collate_pairs(batch):
    anchors = torch.stack([b[0] for b in batch])
    positives = torch.stack([b[1] for b in batch])
    ids = [b[2] for b in batch]
    return anchors, positives, ids


# ── InfoNCE with same-invader masking + optional extra hard negs ──
def info_nce_loss(anchor_emb, pos_emb, ids, temperature=TEMPERATURE, extra_neg_emb=None):
    B = anchor_emb.shape[0]
    logits = anchor_emb @ pos_emb.T                       # (B, B) cosine
    # mask same-invader off-diagonals (they are positives, not negatives)
    id_idx = {v: i for i, v in enumerate(sorted(set(ids)))}
    id_t = torch.tensor([id_idx[x] for x in ids], device=anchor_emb.device)
    same = id_t[:, None] == id_t[None, :]
    eye = torch.eye(B, dtype=torch.bool, device=anchor_emb.device)
    logits = logits.masked_fill(same & ~eye, float('-inf'))
    if extra_neg_emb is not None and extra_neg_emb.numel() > 0:
        logits = torch.cat([logits, anchor_emb @ extra_neg_emb.T], dim=1)
    logits = logits / temperature
    targets = torch.arange(B, device=anchor_emb.device)
    return F.cross_entropy(logits, targets)


# ── Retrieval evaluation: top-k recall, flash -> reference ───────
@torch.no_grad()
def embed_references(backbone, head, by_id, processor, detector, device, batch=64):
    """Mean-embed each invader's reference crops -> one vector per invader."""
    backbone.eval()
    if head: head.eval()
    items = [(iid, p) for iid, paths in by_id.items() for p in paths]
    embs, keys = [], []
    for s in range(0, len(items), batch):
        chunk = items[s:s+batch]
        imgs = []
        for iid, p in chunk:
            c = (detector.crop(IMAGE_ROOT / p) if detector else None) or open_image(IMAGE_ROOT / p)
            imgs.append(c)
        pv = processor(images=imgs, return_tensors='pt')['pixel_values'].to(device)
        e = encode(backbone, head, pv).cpu()
        embs.append(e); keys.extend(iid for iid, _ in chunk)
    embs = torch.cat(embs)
    # mean per invader, then renormalize
    by = {}
    for k, e in zip(keys, embs):
        by.setdefault(k, []).append(e)
    ids = list(by.keys())
    mat = F.normalize(torch.stack([torch.stack(by[i]).mean(0) for i in ids]), dim=-1)
    return mat, ids


@torch.no_grad()
def flash_recall(backbone, head, flash_pairs, ref_mat, ref_ids, processor, detector, device,
                 ks=(1, 5, 10), batch=64):
    """Recall@k for flash images against the per-invader reference matrix."""
    backbone.eval()
    if head: head.eval()
    if not flash_pairs:
        return {k: float('nan') for k in ks}
    q_emb, q_ids = [], []
    for s in range(0, len(flash_pairs), batch):
        chunk = flash_pairs[s:s+batch]
        imgs = []
        for path, _ in chunk:
            c = (detector.crop(path) if detector else None) or open_image(path)
            imgs.append(c)
        pv = processor(images=imgs, return_tensors='pt')['pixel_values'].to(device)
        q_emb.append(encode(backbone, head, pv).cpu())
        q_ids.extend(iid for _, iid in chunk)
    q_emb = torch.cat(q_emb)
    sims = q_emb @ ref_mat.T
    topk = sims.topk(max(ks), dim=1).indices
    out = {}
    for k in ks:
        hits = sum(q_ids[q] in {ref_ids[j] for j in topk[q, :k].tolist()}
                   for q in range(q_emb.shape[0]))
        out[k] = hits / q_emb.shape[0]
    return out


def split_by_invader(rows, val_fraction=VAL_FRACTION):
    by_id = {}
    for r in rows:
        by_id.setdefault(r['invader_id'], []).append(r['image_path'])
    trainable = [k for k, v in by_id.items() if len(v) >= 2]
    random.seed(42); random.shuffle(trainable)
    n_val = max(1, int(len(trainable) * val_fraction))
    val_ids = set(trainable[:n_val])
    train = {k: v for k, v in by_id.items() if k not in val_ids and len(v) >= 2}
    val   = {k: v for k, v in by_id.items() if k in val_ids}
    return train, val

print('Code loaded')


In [ ]:
# ── Load manifests, build datasets, model, optimizer ─────────────
rows = [json.loads(l) for l in MANIFEST.read_text().splitlines() if l.strip()]
train_inv, val_inv = split_by_invader(rows)
print(f'Train invaders: {len(train_inv)}  Val invaders: {len(val_inv)}')

# Flash pairs + per-image hard negatives (verify-UI rejections)
flash_train_pairs, flash_val_pairs = [], []
flash_hard_negatives = {}   # flash_path -> [rejected_mosaic_id, ...]
if FLASH_LABELS.exists():
    flash_rows = [json.loads(l) for l in FLASH_LABELS.read_text().splitlines() if l.strip()]
    all_pairs = []
    for r in flash_rows:
        path = FLASH_ROOT / f"flash/{Path(r['image_path']).name}"
        if path.exists() and r['invader_id'] in train_inv:
            ps = str(path)
            all_pairs.append((ps, r['invader_id']))
            rej = [m for m in r.get('rejected_candidates', [])
                   if m != r['invader_id'] and m in train_inv]
            if rej:
                flash_hard_negatives[ps] = rej[:HARDNEG_MAX_PER_IMG]
    random.seed(99); random.shuffle(all_pairs)
    n_val = max(1, int(len(all_pairs) * 0.15))
    flash_val_pairs   = all_pairs[:n_val]
    flash_train_pairs = all_pairs[n_val:]
    print(f'Flash: {len(flash_train_pairs)} train, {len(flash_val_pairs)} val; '
          f'{len(flash_hard_negatives)} images have hard negs')
else:
    print('No flash_labels.jsonl found')

# Detector
detector = MosaicDetector(DETECTOR_PATH) if DETECTOR_PATH.exists() else None
print('Detector loaded' if detector else 'No detector — using full images')

# Model
model_source = WARMSTART_FROM or BASE_MODEL
processor = AutoImageProcessor.from_pretrained(model_source)
backbone  = AutoModel.from_pretrained(model_source).to(device)

if PROJ_HEAD:
    for p in backbone.parameters():
        p.requires_grad = False
    backbone.eval()
    head = ProjectionHead(in_dim=backbone.config.hidden_size, out_dim=PROJ_DIM).to(device)
    params = head.parameters()
    optimizer = torch.optim.AdamW(params, lr=LR)
else:
    head = None
    for p in backbone.parameters():
        p.requires_grad = False
    for block in backbone.encoder.layer[-UNFREEZE_N:]:
        for p in block.parameters():
            p.requires_grad = True
    for p in backbone.layernorm.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, backbone.parameters()), lr=BACKBONE_LR)

n_train = sum(p.numel() for p in (head.parameters() if PROJ_HEAD
              else (p for p in backbone.parameters() if p.requires_grad)))
print(f'Trainable params: {n_train:,}')

train_ds = PairDataset(train_inv, processor, samples_per_invader=4, augment=True,
                       detector=detector, flash_pairs=flash_train_pairs,
                       flash_oversample=FLASH_OVERSAMPLE)
print(f'Train items/epoch: {len(train_ds)} ({train_ds._ref_len} ref + {train_ds._flash_len} flash)')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_pairs, drop_last=True)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS)
print('Ready to train')


In [ ]:
# ── Training loop ────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def build_hardneg_emb(flash_hard_negs_for_batch_ids, ref_lookup_mat, ref_lookup_ids,
                      pos_lookup, drop_cos=HARDNEG_DROP_COS):
    """Turn rejected mosaic_ids into a clean (M, D) negative embedding bank.

    Cleaning: drop any hard neg whose reference embedding is too close to its
    own positive's reference embedding (likely a near-twin / mislabel that would
    poison the loss). Returns a single shared negative bank for the batch.
    """
    id_to_row = {iid: i for i, iid in enumerate(ref_lookup_ids)}
    keep = set()
    for flash_path, pos_id in pos_lookup.items():
        if pos_id not in id_to_row:
            continue
        pos_vec = ref_lookup_mat[id_to_row[pos_id]]
        for neg_id in flash_hard_negs_for_batch_ids.get(flash_path, []):
            if neg_id not in id_to_row:
                continue
            cos = float(pos_vec @ ref_lookup_mat[id_to_row[neg_id]])
            if cos < drop_cos:
                keep.add(neg_id)
    rows_ = [id_to_row[i] for i in keep if i in id_to_row]
    if not rows_:
        return None
    return ref_lookup_mat[rows_].to(device)

best_recall = -1.0
history = []

for epoch in range(1, TOTAL_EPOCHS + 1):
    t0 = time.time()
    if not PROJ_HEAD:
        backbone.train()
    if head: head.train()

    use_hardneg = epoch >= HARDNEG_START_EPOCH and bool(flash_hard_negatives)
    # Refresh the reference embedding bank for hard-neg cleaning (cheap, once/epoch)
    extra_neg_emb = None
    if use_hardneg:
        ref_mat_clean, ref_ids_clean = embed_references(
            backbone, head, train_inv, processor, detector, device)
        pos_lookup = {p: i for p, i in flash_train_pairs}
        extra_neg_emb = build_hardneg_emb(flash_hard_negatives, ref_mat_clean,
                                          ref_ids_clean, pos_lookup)

    train_loss = 0.0
    for anchor, positive, ids in train_loader:
        anchor, positive = anchor.to(device), positive.to(device)
        a = encode(backbone, head, anchor)
        p = encode(backbone, head, positive)
        loss = info_nce_loss(a, p, ids, temperature=TEMPERATURE,
                             extra_neg_emb=extra_neg_emb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    scheduler.step()

    # ── Retrieval eval: this is what we actually optimize for ──
    ref_mat, ref_ids = embed_references(backbone, head, train_inv, processor, detector, device)
    rec = flash_recall(backbone, head, flash_val_pairs, ref_mat, ref_ids,
                       processor, detector, device, ks=(1, 5, 10))

    elapsed = time.time() - t0
    hn_tag = ' +hardneg' if use_hardneg else ''
    print(f'Epoch {epoch:3d}/{TOTAL_EPOCHS}  loss={train_loss:.4f}  '
          f'R@1={rec[1]:.3f} R@5={rec[5]:.3f} R@10={rec[10]:.3f}  {elapsed:.0f}s{hn_tag}')
    history.append({'epoch': epoch, 'loss': train_loss, **{f'recall@{k}': v for k, v in rec.items()}})

    if rec[1] > best_recall:
        best_recall = rec[1]
        if PROJ_HEAD:
            torch.save(head.state_dict(), OUTPUT_DIR / 'projection_head.pt')
            processor.save_pretrained(OUTPUT_DIR)
            (OUTPUT_DIR / 'head_config.json').write_text(json.dumps(
                {'in_dim': backbone.config.hidden_size, 'out_dim': PROJ_DIM,
                 'base_model': BASE_MODEL}))
        else:
            backbone.save_pretrained(OUTPUT_DIR)
            processor.save_pretrained(OUTPUT_DIR)
        (OUTPUT_DIR / 'training_state.json').write_text(json.dumps(
            {'best_recall@1': best_recall, 'epoch': epoch}))
        print(f'  -> saved (best R@1={best_recall:.3f})')

(OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2))
print(f'\nDone. Best R@1: {best_recall:.3f}')
print(f'Weights at: {OUTPUT_DIR}')


In [ ]:
# ── Optional: plot recall curve ──────────────────────────────────
import json
import matplotlib.pyplot as plt
hist = json.loads((OUTPUT_DIR / 'history.json').read_text())
ep = [h['epoch'] for h in hist]
plt.figure(figsize=(8,4))
for k in (1,5,10):
    plt.plot(ep, [h[f'recall@{k}'] for h in hist], marker='.', label=f'R@{k}')
plt.axvline(HARDNEG_START_EPOCH, ls='--', c='gray', label='hard negs on')
plt.xlabel('epoch'); plt.ylabel('recall'); plt.ylim(0,1.02); plt.legend(); plt.grid(alpha=.3)
plt.title(f'{EXP_NAME} — flash→reference retrieval recall'); plt.show()